In [27]:
import pandas as pd 
import numpy as np 

In [28]:
df= pd.read_csv("After_EDA_and_Feature_ENginering.csv")
df.head()

,Brand,Model_Name,Model_URL,Model_Image,5G_Support,Weight_g,Refresh_Rate_Hz,Display_Size_inch,Resolution,PPI_Density,...,Fingerprint_Position,Has_Face_Unlock,Has_Gyro,Has_Compass,Has_Barometer,Sensor_Count,Color_Option_Count,Has_FM_Radio,FM_Has_RDS,FM_Can_Record
0,Acer,Acer Liquid,https://www.gsmarena.com/acer_liquid-2968.php,https://fdn2.gsmarena.com/vv/bigpic/acer-liqui...,No,135.0,60.0,3.5,480x800,267.0,...,NaN,0,0,0,0,2,3,0,0,0
1,Acer,Acer neoTouch,https://www.gsmarena.com/acer_neotouch-2958.php,https://fdn2.gsmarena.com/vv/bigpic/acer-neo-t...,No,130.0,60.0,3.8,480x800,246.0,...,NaN,0,0,0,0,2,1,1,1,0
2,Acer,Acer beTouch E200,https://www.gsmarena.com/acer_betouch_e200-296...,https://fdn2.gsmarena.com/vv/bigpic/acer-be-to...,No,146.0,60.0,3.0,240x400,155.0,...,NaN,0,0,1,0,3,2,0,0,0
3,Acer,Acer beTouch E100,https://www.gsmarena.com/acer_betouch_e100-295...,https://fdn2.gsmarena.com/vv/bigpic/acer-be-to...,No,118.0,60.0,3.2,240x400,146.0,...,NaN,0,0,1,0,3,1,0,0,0
4,Acer,Acer beTouch E101,https://www.gsmarena.com/acer_betouch_e101-296...,https://fdn2.gsmarena.com/vv/bigpic/acer-be-to...,No,118.0,60.0,3.2,240x400,146.0,...,NaN,0,0,1,0,3,1,0,0,0


In [29]:
df.isnull().mean().sort_values(ascending=False)

Display_Max_Brightness_nits    0.722874
Fingerprint_Position           0.454998
Density_g_per_cm3              0.000237
Flash_Type                     0.000119
Brand                          0.000000
                                 ...   
Sensor_Count                   0.000000
Color_Option_Count             0.000000
Has_FM_Radio                   0.000000
FM_Has_RDS                     0.000000
FM_Can_Record                  0.000000
Length: 152, dtype: float64

In [30]:
df=df.drop(columns=["Fingerprint_Position","Display_Max_Brightness_nits" ,"Model_URL","Model_Image"])

In [31]:
df['Density_g_per_cm3'] = ( df['Weight_g'] / df['Volume_cm3'])

In [32]:
df['Flash_Type'] = (df.groupby('Brand', dropna=False)['Flash_Type'].transform(lambda x: x.fillna(x.mode().iloc[0]) if not x.mode().empty else x))
df['Flash_Type'] = df['Flash_Type'].fillna('None')

In [33]:
df.columns

Index(['Brand', 'Model_Name', '5G_Support', 'Weight_g', 'Refresh_Rate_Hz',
       'Display_Size_inch', 'Resolution', 'PPI_Density', 'Screen_to_Body_Pct',
       'Display_Protection',
       ...
       'Has_Fingerprint', 'Has_Face_Unlock', 'Has_Gyro', 'Has_Compass',
       'Has_Barometer', 'Sensor_Count', 'Color_Option_Count', 'Has_FM_Radio',
       'FM_Has_RDS', 'FM_Can_Record'],
      dtype='str', length=148)

In [34]:
provenance_cols = [c for c in df.columns if c.endswith('_Source') or c.endswith('_is_imputed')]
df = df.drop(columns=provenance_cols)
print("Dropped:", provenance_cols)

Dropped: ['AnTuTu_Score_Source', 'GeekBench_Score_Source', 'GPU_Source', 'Chipset_Source', 'Price_INR_Source', 'Main_Camera_MP_Source', 'Selfie_Camera_MP_Source', 'Storage_GB_Source', 'RAM_GB_Source', 'Lens_Count_Source', 'Battery_mAh_Source', 'Display_Type_Source', 'AnTuTu_Score_is_imputed', 'GeekBench_Score_is_imputed', 'GPU_is_imputed', 'Chipset_is_imputed', 'Price_INR_is_imputed', 'Main_Camera_MP_is_imputed', 'Selfie_Camera_MP_is_imputed', 'Storage_GB_is_imputed', 'RAM_GB_is_imputed', 'Lens_Count_is_imputed', 'Battery_mAh_is_imputed', 'Display_Type_is_imputed', 'Price_EUR_is_imputed', 'Wired_Charging_W_is_imputed', 'CPU_architecture_is_imputed', 'Storage_Performance_Score_is_imputed']


In [35]:
df.duplicated().sum()

np.int64(29)

In [36]:
df.drop_duplicates(inplace=True)

In [37]:
df.info()

<class 'pandas.DataFrame'>
Index: 8404 entries, 0 to 8432
Columns: 120 entries, Brand to FM_Can_Record
dtypes: float64(39), int64(53), str(28)
memory usage: 7.8 MB


In [38]:
categorical_feature = [feature for feature in df.columns if df[feature].dtype in ['object', 'category', 'string']]
Numeerical_feature = [feature for feature in df.columns if df[feature].dtype in ["float64"]]

In [39]:
import numpy as np

for col in Numeerical_feature:
    n_inf = np.isinf(df[col]).sum()
    if n_inf > 0:
        print(f"{col}: {n_inf} infinite values")
    n_huge = (df[col].abs() > 1e15).sum()
    if n_huge > 0:
        print(f"{col}: {n_huge} suspiciously huge values")

Density_g_per_cm3: 2 infinite values
Density_g_per_cm3: 2 suspiciously huge values


In [40]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics.pairwise import cosine_similarity
import joblib
import numpy as np

df[Numeerical_feature] = df[Numeerical_feature].replace([np.inf, -np.inf], np.nan)

df = df.sort_values('Price_EUR').drop_duplicates(subset=['Brand', 'Model_Name'], keep='first')
df = df.reset_index(drop=True)

categorical_feature = [c for c in categorical_feature if c != 'Model_Name'] 

In [41]:
feature_weights = {
    'Chipset_Generation': 2.0,
    'CPU_max_clock_ghz': 1.8,
    'GPU_Is_Flagship': 1.8,
    'RAM_GB': 1.5,
    'Storage_GB': 1.3,
    'Battery_mAh': 1.3,
    'AnTuTu_Score': 1.6,
    'GeekBench_Score': 1.4,
    'Color_Option_Count': 0.3,
    'FM_Has_RDS': 0.2,
    'FM_Can_Record': 0.2,
}

weight_vector = np.array([feature_weights.get(f, 1.0) for f in Numeerical_feature])

def apply_weights(X):
    return X * weight_vector

In [42]:
numeric_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
    ('weight', FunctionTransformer(apply_weights)),
])

In [43]:
categorical_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encode', OneHotEncoder(handle_unknown='ignore')),
])

In [44]:
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, Numeerical_feature),
    ('cat', categorical_pipeline, categorical_feature),
])

In [45]:
full_pipeline = Pipeline([('preprocessor', preprocessor)])

In [46]:
phone_matrix = full_pipeline.fit_transform(df)
similarity_matrix = cosine_similarity(phone_matrix)

print("Phone matrix shape:", phone_matrix.shape)
print("Similarity matrix shape:", similarity_matrix.shape)

Phone matrix shape: (8357, 1257)
Similarity matrix shape: (8357, 8357)


In [47]:
def similar_phones(model_name, top_n=5):
    matches = df.index[df['Model_Name'] == model_name]
    if len(matches) == 0:
        return f"'{model_name}' not found in dataset"
    idx = matches[0]
    sims = similarity_matrix[idx]
    order = [i for i in np.argsort(sims)[::-1] if i != idx][:top_n]
    result = df.iloc[order][['Brand', 'Model_Name', 'Price_EUR']].copy()
    result['Similarity'] = sims[order].round(3)
    return result

In [48]:
similar_phones("ZTE nubia RedMagic 11S Pro")

,Brand,Model_Name,Price_EUR,Similarity
8079,Zte,ZTE nubia RedMagic 11S Pro (China),700.00,0.980
8143,Zte,ZTE nubia RedMagic 11S Pro+,780.00,0.978
7427,Zte,ZTE nubia RedMagic 11 Air,430.00,0.888
8091,Zte,ZTE nubia RedMagic 11 Pro,728.63,0.877
7811,Oneplus,OnePlus 15R,534.00,0.866


In [49]:
similar_phones("Samsung Galaxy S25 Ultra")

,Brand,Model_Name,Price_EUR,Similarity
8237,Samsung,Samsung Galaxy S26 Ultra,934.99,0.991
7935,Samsung,Samsung Galaxy S25 Edge,603.79,0.974
7829,Samsung,Samsung Galaxy S24 Ultra,548.13,0.970
8103,Samsung,Samsung Galaxy S25+,735.00,0.955
8196,Samsung,Samsung Galaxy S26+,829.00,0.949


In [50]:
similar_phones("Apple iPhone 17 Pro Max")

,Brand,Model_Name,Price_EUR,Similarity
8275,Apple,Apple iPhone 17 Pro,1117.46,0.993
8215,Apple,Apple iPhone 16 Pro Max,879.99,0.967
8084,Apple,Apple iPhone 17,718.99,0.960
8090,Apple,Apple iPhone Air,728.09,0.950
8080,Apple,Apple iPhone 16 Pro,709.00,0.945


In [51]:
duplicate_counts = df['Model_Name'].value_counts()
print(duplicate_counts[duplicate_counts > 1])

duplicates = df[df['Model_Name'].duplicated(keep=False)].sort_values('Model_Name')

print(f"Total duplicate rows: {df['Model_Name'].duplicated(keep=False).sum()}")
print(f"Unique model names with duplicates: {(duplicate_counts > 1).sum()}")

Series([], Name: count, dtype: int64)
Total duplicate rows: 0
Unique model names with duplicates: 0


In [52]:
joblib.dump({
    'pipeline': full_pipeline,
    'similarity_matrix': similarity_matrix,
    'df': df,
    'feature_weights': feature_weights,
}, 'similarity_bundle.joblib')

print("Saved: similarity_bundle.joblib")

Saved: similarity_bundle.joblib
